# Graph Visualization

This notebook demonstrates GestaltDB's packaged offline visualization: self-contained interactive `.html` artifacts (D3.js + React, prebuilt and shipped with the library) rendered from node/edge collections, Cypher results, typed samples, and ML sampler batches. No JavaScript toolchain or network access is needed — the last expression of each section renders inline in Jupyter.

## Local Development Import

When running from the repository checkout, add `../src` to `sys.path`. If the package is installed, this cell is harmless.

In [ ]:
#@title Installs
INSTALL_GESTALTDB = False  #@param {type:"boolean"}
if INSTALL_GESTALTDB:
  !pip install git+https://github.com/mylonasc/gestaltdb.git


## Build a Small Biomedical Graph

In [ ]:
import shutil
import tempfile

from gestaltdb.graphdb import Edge, GraphDB, Node
from gestaltdb.kvstores import LevelDBStore
from gestaltdb.serializers import PickleSerializer

db_path = tempfile.mkdtemp(prefix="gestaltdb_viz_")
graph = GraphDB(LevelDBStore(path=db_path), PickleSerializer())

graph.put_nodes([
    Node(node_id="drug-1", labels=["Drug"], properties={"name": "Aspirin", "score": 0.9}),
    Node(node_id="drug-2", labels=["Drug"], properties={"name": "Ibuprofen", "score": 0.7}),
    Node(node_id="protein-1", labels=["Protein"], properties={"name": "COX-1"}),
    Node(node_id="protein-2", labels=["Protein"], properties={"name": "COX-2"}),
    Node(node_id="disease-1", labels=["Disease"], properties={"name": "Inflammation"}),
])
graph.put_edges_bulk([
    Edge(edge_id="d1-p1", source="drug-1", target="protein-1", properties={"type": "binds"}),
    Edge(edge_id="d1-p2", source="drug-1", target="protein-2", properties={"type": "binds"}),
    Edge(edge_id="d2-p2", source="drug-2", target="protein-2", properties={"type": "binds"}),
    Edge(edge_id="p2-dis1", source="protein-2", target="disease-1", properties={"type": "associated_with"}),
])
print("nodes:", sorted(n.get_id for n in graph.nodes_by_label("Drug")))

## Visualize Explicit Nodes and Edges

`visualize_nodes_edges` renders any `Node`/`Edge` collections. `.save(path)` writes a shareable offline `.html` file; the bare `figure` expression renders inline below.

In [ ]:
from gestaltdb.viz.api import visualize_nodes_edges

nodes = [graph.get_node(nid) for nid in (b"drug-1", b"protein-1", b"protein-2", b"disease-1")]
edges = [graph.get_edge(eid) for eid in (b"d1-p1", b"d1-p2", b"p2-dis1")]
figure = visualize_nodes_edges(nodes, edges)
figure.save(f"{db_path}/binds.html")
print(repr(figure))
figure

## Visualize a Cypher Query

Matched entities are highlighted in the canvas; the rest dims but stays navigable. `GraphDB.visualize` is the same path as a method.

In [ ]:
cypher_figure = graph.visualize(
    "MATCH (d:Drug)-[r:binds]->(p:Protein) RETURN d, r, p"
)
cypher_figure.save(f"{db_path}/cypher.html")
print(repr(cypher_figure))
cypher_figure

## Sample Large Neighborhoods Instead of Dumping Them

`visualize_sample` follows a typed `SamplingPattern` from seed nodes, so even 100k-node graphs stay interactive.

In [ ]:
from gestaltdb.sampling import SamplingHop, SamplingPattern
from gestaltdb.viz.api import visualize_sample

pattern = SamplingPattern([
    SamplingHop("binds", direction="out", sample_size=5),
    SamplingHop("associated_with", direction="out", sample_size=3),
])
sample_figure = visualize_sample(graph, ["drug-1"], pattern)
sample_figure.save(f"{db_path}/sample.html")
print(repr(sample_figure))
sample_figure

## Options, Themes, and Caps

`VizOptions` controls caps, the absolute ceiling, theme, force-layout physics, and label visibility. Caps truncate deterministically with a `TruncationWarning` and an exact on-canvas banner.

In [ ]:
import warnings

from gestaltdb.viz.api import VizOptions
from gestaltdb.viz.ir import TruncationWarning

options = VizOptions(max_nodes=3, max_edges=10, theme="dark", title="Capped drug graph")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", TruncationWarning)
    capped = graph.visualize("MATCH (n) RETURN n", options=options)
print(repr(capped))
print(capped.viz.truncation.to_dict())
capped

## Inspect ML Sampler Batches

`visualize_sampler_batch` maps local batch rows back through `node_ids_global` to external IDs for training-pipeline inspection.

In [ ]:
from gestaltdb.sampling import SamplerEngine
from gestaltdb.viz.api import visualize_sampler_batch

snapshot = graph.build_sampler_snapshot(f"{db_path}/snap")
engine = SamplerEngine.load(snapshot.path, mode="ram", seed=13)
seed_edge = snapshot.external_edge_ids.tolist().index("d1-p1")
batch = engine.sample_subgraph([seed_edge], fanouts=[5])
batch_figure = visualize_sampler_batch(batch, snapshot)
batch_figure.save(f"{db_path}/batch.html")
print(repr(batch_figure))
batch_figure

## Close the Store

In [ ]:
graph.close()
shutil.rmtree(db_path, ignore_errors=True)
print("closed")